# Free SDXL LoRA Training: rexstyle
This notebook trains a custom **SDXL LoRA** for free on a Google Colab T4 GPU (~15GB VRAM).
- **Trigger Word**: `rexstyle`
- **Resolution**: `1024x1024`
- **LoRA Rank (Network Dim)**: `16`
- **Total Steps**: `1200` (Checkpoints saved every `250` steps)
- **Base Model**: `stabilityai/stable-diffusion-xl-base-1.0`

> Make sure to select **Runtime > Change runtime type > T4 GPU** before running.

In [ ]:
# 1. Verify GPU Allocation
!nvidia-smi

In [ ]:
# 2. Clone Kohya sd-scripts & Install Dependencies
!git clone https://github.com/kohya-ss/sd-scripts.git
%cd sd-scripts
!pip install -q torch==2.1.2+cu121 torchvision==0.16.2+cu121 --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q xformers==0.0.23.post1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q accelerate diffusers transformers safetensors ftfy einops bitsandbytes
%cd ..

In [ ]:
# 3. Unpack Training Kit Dataset
import os, shutil, zipfile
from google.colab import files

dataset_dir = '/content/dataset/img/10_rexstyle'
os.makedirs(dataset_dir, exist_ok=True)

if not os.path.exists('/content/training_kit.zip'):
    print('Please upload your training_kit.zip generated by linkedin-image-agent:')
    uploaded = files.upload()

with zipfile.ZipFile('/content/training_kit.zip', 'r') as zf:
    zf.extractall(dataset_dir)

print(f'Successfully unpacked {len(os.listdir(dataset_dir))} files into {dataset_dir}')

In [ ]:
# 4. Launch SDXL LoRA Training Job
!accelerate launch --num_cpu_threads_per_process=2 sd-scripts/sdxl_train_network.py \
    --pretrained_model_name_or_path='stabilityai/stable-diffusion-xl-base-1.0' \
    --train_data_dir='/content/dataset/img' \
    --output_dir='/content/output' \
    --output_name='rexstyle_sdxl_lora' \
    --resolution=1024,1024 \
    --train_batch_size=1 \
    --max_train_steps=1200 \
    --save_every_n_steps=250 \
    --network_module='networks.lora' \
    --network_dim=16 \
    --network_alpha=16 \
    --learning_rate=1e-4 \
    --optimizer_type='AdamW8bit' \
    --lr_scheduler='cosine' \
    --mixed_precision='fp16' \
    --save_precision='fp16' \
    --gradient_checkpointing \
    --xformers \
    --cache_latents

In [ ]:
# 5. Test Inference with LoRA
import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    variant='fp16',
).to('cuda')

pipe.load_lora_weights('/content/output/rexstyle_sdxl_lora.safetensors')
prompt = 'rexstyle, tactile retro-futuristic CRT computer terminal on a cream desk, high contrast, studio lighting'
image = pipe(prompt=prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
image.save('/content/test_sample.jpg')
image

In [ ]:
# 6. Download Trained LoRA Weights
from google.colab import files
files.download('/content/output/rexstyle_sdxl_lora.safetensors')
print('Place this file into linkedin-image-agent/models/rexstyle_sdxl_lora.safetensors')